In [1]:
import numpy as np
import pandas as pd

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Regression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Hyperparameter tuning
from sklearn.model_selection import GridSearchCV

In [11]:
df = pd.read_csv("StudentsPerformance.csv")

In [12]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race/ethnicity               1000 non-null   object
 2   parental level of education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test preparation course      1000 non-null   object
 5   math score                   1000 non-null   int64 
 6   reading score                1000 non-null   int64 
 7   writing score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB


In [14]:
df.isnull().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [16]:
X = df.drop("math score", axis=1)
y = df["math score"]

In [17]:
# Convert categorical columns into numerical
X = pd.get_dummies(X, drop_first=True)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [20]:
# Feature Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Ridge Regression

In [22]:
from sklearn.linear_model import Ridge

ridge = Ridge()
ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_test)

In [23]:
ridge = Ridge()

ridge.fit(X_train, y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


# Evaluate Ridge Regression

In [24]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Ridge Regression Results")
print("------------------------")
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

Ridge Regression Results
------------------------
MAE : 4.2128668003063074
MSE : 29.067898878597706
RMSE: 5.391465373958893
R2 Score: 0.8805453685953483


# Hyperparameter Tuning (Ridge)

In [25]:
from sklearn.model_selection import GridSearchCV

params = {
    'alpha':[0.01,0.1,1,10,100]
}

grid = GridSearchCV(
    Ridge(),
    params,
    cv=5
)

grid.fit(X_train,y_train)

print("Best Parameters:",grid.best_params_)
print("Best Score:",grid.best_score_)

best_ridge = grid.best_estimator_

ridge_pred = best_ridge.predict(X_test)

print("Tuned R2:",r2_score(y_test,ridge_pred))

Best Parameters: {'alpha': 1}
Best Score: 0.8685926351918326
Tuned R2: 0.8805453685953483


# Logistic Regression

In [26]:
df["Pass"] = (df["math score"] >= 50).astype(int)

X = df.drop(["math score","Pass"],axis=1)

X = pd.get_dummies(X,drop_first=True)

y = df["Pass"]

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train Logistic Regression

In [27]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train,y_train)

pred = log_model.predict(X_test)

# Evaluate Logistic Regression

In [28]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

print("Logistic Regression Results")
print("---------------------------")

print("Accuracy :",accuracy_score(y_test,pred))
print("Precision:",precision_score(y_test,pred))
print("Recall   :",recall_score(y_test,pred))
print("F1 Score :",f1_score(y_test,pred))

Logistic Regression Results
---------------------------
Accuracy : 0.895
Precision: 0.9190751445086706
Recall   : 0.9578313253012049
F1 Score : 0.9380530973451328


# GridSearchCV for Logistic Regression

In [29]:
params = {
    "C":[0.01,0.1,1,10],
    "solver":["lbfgs","liblinear"]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    params,
    cv=5
)

grid.fit(X_train,y_train)

print("Best Parameters:",grid.best_params_)

best_log = grid.best_estimator_

pred = best_log.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))
print("Precision:",precision_score(y_test,pred))
print("Recall:",recall_score(y_test,pred))
print("F1:",f1_score(y_test,pred))

Best Parameters: {'C': 10, 'solver': 'lbfgs'}
Accuracy: 0.9
Precision: 0.9244186046511628
Recall: 0.9578313253012049
F1: 0.9408284023668639


# K-Means Clustering

In [31]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_data = df[[
    "math score",
    "reading score",
    "writing score"
]]

scaler = StandardScaler()

cluster_scaled = scaler.fit_transform(cluster_data)

kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

kmeans.fit(cluster_scaled)

labels = kmeans.labels_

C:\Users\22053\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


In [ ]:
print("Silhouette Scores")

for k in range(2,11):

    model = KMeans(
        n_clusters=k,
        random_state=42
    )

    model.fit(cluster_scaled)

    score = silhouette_score(
        cluster_scaled,
        model.labels_
    )

    print("K =",k," Score =",score)

Silhouette Scores
K = 2  Score = 0.47410805799442546
K = 3  Score = 0.40599504065325176


C:\Users\22053\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\22053\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\22053\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


K = 4  Score = 0.3530512639720019
K = 5  Score = 0.33065197797579354
K = 6  Score = 0.31927905215476116
K = 7  Score = 0.2984423997584124


C:\Users\22053\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\22053\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\22053\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\22053\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Window